In [ ]:
'''
acvmg: NOTAS SOBRE O DATASET !!!

Foi difícil encontrar essa informação.

O dataset está dividido em:
- F0: hélice saudável
- F1, F2, F3: condições danificadas 
- SV0: saudável
- SV1, SV3: níveis de severidade
- SP: velocidade
- C: canal do acelerômetro (?)
- R1 - R3 indicam que são diferentes gravações das mesmas condições.

F1 - F3 são simplesmente diferentes tipos de falhas. Não consegui encontrar qual falha, mas são falhas.
em nível de severidade, SV0 é saudável, e SV1 < SV2 < SV3 em nível de severidade
SP consiste em diferentes configurações de velocidade para o drone, SP1 < SP2 < SP3
-> para comparações justas, deveremos manter SP constante ...

'''

'\nacvmg: NOTAS SOBRE O DATASET !!!\n\n\n'

In [ ]:
def buildDataset():
    from pathlib import Path
    from scipy.io import loadmat
    import numpy as np

    dados = Path("../../data")

    dataset = []

    for file in sorted(dados.rglob("*.mat")):
        data = loadmat(file, simplify_cells=True)

        nome = str(file.name)   # o nome do arquivo contêm as labels que eu quero pra montar o dataset

        nome = nome.split("_")
        nome[-1] = nome[-1].split(".")

        #print(nome)

        condition, severity, speed, channel, recording = None, None, None, None, None

        for i in range(len(nome)):
            if nome[0] == 'F0':
                condition = 0  
            elif nome[0] == 'F1':
                condition = 1
            elif nome[0] == 'F2':
                condition = 2
            elif nome[0] == 'F3':
                condition = 3

            if nome[1] == 'SV0':
                severity = 0
            elif nome[1] == 'SV1':
                severity = 1
            elif nome[1] == 'SV2':
                severity = 2
            elif nome[1] == 'SV3':
                severity = 3

            if nome[2] == 'SP0':
                speed = 0
            elif nome[2] == 'SP1':
                speed = 1
            elif nome[2] == 'SP2':
                speed = 2
            elif nome[2] == 'SP3':
                speed = 3

            if nome[3] == 'C1':
                channel = 1
            elif nome[3] == 'C2':
                channel = 2
            elif nome[3] == 'C3':
                channel = 3
            elif nome[3] == 'C4':
                channel = 4

            try:
                if nome[4][0] == 'R1':
                    recording = 1
                elif nome[4][0] == 'R2':
                    recording = 2
                elif nome[4][0] == 'R3':
                    recording = 3
            except:
                recording = 1

                if nome[3][0] == 'C1':
                    channel = 1
                elif nome[3][0] == 'C2':
                    channel = 2
                elif nome[3][0] == 'C3':
                    channel = 3
                elif nome[3][0] == 'C4':
                    channel = 4

        #checks de segurança:
        if condition is not None and severity is not None and speed is not None and channel is not None and recording is not None:
            pass
        else:
            print(f"dados inválidos em: {file.name}")
            #print(condition, severity, speed, channel, recording)
            continue

        acc = data['AccData']       #acelerômetro

        sampling_rate = acc['samplingRate']
        timestamps    = acc['t']
        signal        = acc['Vib_Data']

        data = [signal, sampling_rate, timestamps, condition, severity, speed, channel, recording]
        dataset.append(data)
    
    return dataset
        
dataset = buildDataset()

### TABELA:

Foram selecionados todos os registros saudáveis (`F0`), com severidade `SV0`, velocidade `SP1` e canal do acelerômetro `C1`.

| Registro | Condição | Severidade | Velocidade (`SP`) | Canal (`C`) | Repetição | Amostras | Frequência | Duração | Por que podem ser comparados |
|---|---|---:|---:|---:|---:|---:|---:|---:|---|
| `F0_SV0_SP1_C1_R1.mat` | Saudável | 0 | 1 | 1 | 1 | 600.000 | 10.000 Hz | 60 s | Mesmas condições experimentais |
| `F0_SV0_SP1_C1_R2.mat` | Saudável | 0 | 1 | 1 | 2 | 600.000 | 10.000 Hz | 60 s | Mesmas condições experimentais |
| `F0_SV0_SP1_C1_R3.mat` | Saudável | 0 | 1 | 1 | 3 | 600.000 | 10.000 Hz | 60 s | Mesmas condições experimentais |

Os registros podem ser comparados porque mantêm fixos a condição da hélice, a severidade, a velocidade de operação, o canal do acelerômetro, a frequência de amostragem, a quantidade de amostras e a duração. A única diferença é a repetição da medição.